# Улучшенный hybrid ranker для resume-vacancy matching



In [1]:
!pip install pandas numpy scikit-learn torch sentence-transformers

In [2]:
from __future__ import annotations

import json, re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers.evaluation import BinaryClassificationEvaluator
from sentence_transformers.cross_encoder.evaluation import CrossEncoderClassificationEvaluator
from torch.utils.data import DataLoader


/tmp/ipykernel_2895/1447878474.py:12: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses
/tmp/ipykernel_2895/1447878474.py:14: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import BinaryClassificationEvaluator


In [4]:
DATA_DIR = Path('data')
ARTIFACTS_DIR = Path('artifacts')
BI_DIR = ARTIFACTS_DIR / 'bi_encoder'
CROSS_DIR = ARTIFACTS_DIR / 'cross_encoder'

TRAIN_PATH = DATA_DIR / 'train_pairs.csv'
VAL_PATH = DATA_DIR / 'val_pairs.csv'
TEST_PATH = DATA_DIR / 'test_pairs.csv'


In [5]:
def clean_text(text):
    if pd.isna(text):
        return ''
    return ' '.join(str(text).strip().split())

def validate_hf_model_dir(model_dir):
    config_path = model_dir / 'config.json'
    if not model_dir.exists():
        raise FileNotFoundError(f'Не найдена папка модели: {model_dir}')
    if not config_path.exists():
        raise FileNotFoundError(f'Не найден config.json: {model_dir}')
    with open(config_path, 'r', encoding='utf-8') as f:
        config = json.load(f)
    if 'model_type' not in config:
        raise ValueError(f'В config.json нет model_type: {model_dir}')

def dcg_at_k(relevances, k):
    return float(sum(rel / np.log2(i + 1) for i, rel in enumerate(relevances[:k], start=1)))

def ndcg_at_k(relevances, k):
    ideal = dcg_at_k(sorted(relevances, reverse=True), k)
    return 0.0 if ideal == 0 else dcg_at_k(relevances, k) / ideal

def reciprocal_rank(relevances):
    for i, rel in enumerate(relevances, start=1):
        if rel > 0:
            return 1.0 / i
    return 0.0

def recall_at_k(relevances, total_relevant, k):
    return 0.0 if total_relevant == 0 else sum(relevances[:k]) / total_relevant

def summarize_metrics(details_df, model_name):
    return {
        'model': model_name,
        'Recall@3': float(details_df['Recall@3'].mean()),
        'Recall@5': float(details_df['Recall@5'].mean()),
        'MRR': float(details_df['RR'].mean()),
        'NDCG@5': float(details_df['NDCG@5'].mean()),
    }


## 1. Загрузка данных и моделей

In [6]:
train_df = pd.read_csv(TRAIN_PATH, dtype={'resume_id': str, 'vacancy_id': str})
val_df = pd.read_csv(VAL_PATH, dtype={'resume_id': str, 'vacancy_id': str})
test_df = pd.read_csv(TEST_PATH, dtype={'resume_id': str, 'vacancy_id': str})

print('train:', train_df.shape)
print('val:', val_df.shape)
print('test:', test_df.shape)

def ensure_dir(path):
    path.mkdir(parents=True, exist_ok=True)

def build_bi_train_examples(df):
    positives = df[df['label'] == 1].copy()
    return [
        InputExample(
            texts=[
                f"query: {clean_text(r['resume_text'])}",
                f"passage: {clean_text(r['vacancy_text'])}"
            ]
        )
        for _, r in positives.iterrows()
    ]

def build_bi_evaluator(df):
    return BinaryClassificationEvaluator(
        sentences1=[f"query: {clean_text(x)}" for x in df['resume_text']],
        sentences2=[f"passage: {clean_text(x)}" for x in df['vacancy_text']],
        labels=[int(x) for x in df['label']],
        name='val-binary-cls',
    )

def build_cross_examples(df):
    return [
        InputExample(
            texts=[clean_text(r['resume_text']), clean_text(r['vacancy_text'])],
            label=float(r['label'])
        )
        for _, r in df.iterrows()
    ]

def build_cross_eval(df):
    sentence_pairs = [
        [clean_text(r['resume_text']), clean_text(r['vacancy_text'])]
        for _, r in df.iterrows()
    ]
    labels = [int(x) for x in df['label']]
    return sentence_pairs, labels

if not BI_DIR.exists():
    print('artifacts/bi_encoder не найден, обучаю baseline bi-encoder')
    ensure_dir(BI_DIR)
    bi_model_tmp = SentenceTransformer('intfloat/multilingual-e5-base')
    bi_train_examples = build_bi_train_examples(train_df)
    bi_train_loader = DataLoader(bi_train_examples, shuffle=True, batch_size=16)
    bi_loss = losses.MultipleNegativesRankingLoss(model=bi_model_tmp)
    bi_evaluator = build_bi_evaluator(val_df)
    bi_model_tmp.fit(
        train_objectives=[(bi_train_loader, bi_loss)],
        evaluator=bi_evaluator,
        epochs=3,
        warmup_steps=max(10, len(bi_train_loader)//10),
        output_path=str(BI_DIR),
        show_progress_bar=True,
    )

if not CROSS_DIR.exists():
    print('artifacts/cross_encoder не найден, обучаю baseline cross-encoder')
    ensure_dir(CROSS_DIR)
    cross_model_tmp = CrossEncoder(
        'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1',
        num_labels=1,
        max_length=256,
    )
    cross_train_examples = build_cross_examples(train_df)
    cross_train_loader = DataLoader(cross_train_examples, shuffle=True, batch_size=16)
    val_sentence_pairs, val_labels = build_cross_eval(val_df)
    cross_evaluator = CrossEncoderClassificationEvaluator(
        sentence_pairs=val_sentence_pairs,
        labels=val_labels,
        name='cross-val',
    )
    cross_model_tmp.fit(
        train_dataloader=cross_train_loader,
        evaluator=cross_evaluator,
        epochs=3,
        warmup_steps=max(10, len(cross_train_loader)//10),
        output_path=str(CROSS_DIR),
        show_progress_bar=True,
    )
    cross_model_tmp.model.save_pretrained(str(CROSS_DIR))
    cross_model_tmp.tokenizer.save_pretrained(str(CROSS_DIR))

validate_hf_model_dir(CROSS_DIR)
bi_model = SentenceTransformer(str(BI_DIR))
cross_model = CrossEncoder(str(CROSS_DIR))

train_df.head(2)


train: (1068, 5)
val: (285, 5)
test: (285, 5)
artifacts/bi_encoder не найден, обучаю baseline bi-encoder


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

,resume_id,vacancy_id,resume_text,vacancy_text,label
0,000a4860-4725-11ee-91de-01e20877b34e,f822e22a-0eb1-11ee-8ec2-0d972cff014f,Желаемая должность: Junior-Java-Developer Hard...,Название вакансии: Программист Профессиональна...,1
1,000a4860-4725-11ee-91de-01e20877b34e,744b3700-e3ca-11ef-86ee-d549be31d974,Желаемая должность: Junior-Java-Developer Hard...,Название вакансии: Программист Java Профессион...,0


## 2. Извлечение полей и базовых признаков

In [ ]:
FIELD_ALIASES = {
    'resume_title': ['Желаемая должность', 'Типовая позиция'],
    'vacancy_title': ['Название вакансии'],
    'resume_skills': ['Навыки', 'Hard skills'],
    'vacancy_skills': ['Навыки', 'Hard skills', 'Требования'],
    'resume_exp': ['Опыт'],
    'vacancy_exp': ['Опыт', 'Требования'],
    'resume_region': ['Город', 'Регион'],
    'vacancy_region': ['Регион'],
    'resume_sphere': ['Профсфера', 'Профессиональная сфера'],
    'vacancy_sphere': ['Профессиональная сфера', 'Профсфера'],
}

ALL_FIELD_NAMES = sorted({x for vals in FIELD_ALIASES.values() for x in vals}, key=len, reverse=True)

def extract_field(text, field_names):
    text = clean_text(text)
    field_pat = '|'.join([re.escape(x) for x in ALL_FIELD_NAMES])
    for name in field_names:
        pattern = re.compile(re.escape(name) + r':\s*(.*?)(?=(?:' + field_pat + r')\s*:|$)')
        m = pattern.search(text)
        if m:
            val = m.group(1).strip()
            if val:
                return val
    return ''

STOPWORDS = {'и','в','на','по','с','для','или','от','до','под','над','из','к','a','the','of','to','for','and','or','не','но','что','как'}

def tokenize(text):
    toks = re.findall(r'[A-Za-zА-Яа-я0-9_+#.-]+', clean_text(text).lower())
    return [t for t in toks if len(t) > 1 and t not in STOPWORDS]

def jaccard(a, b):
    a = set(a)
    b = set(b)
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

def overlap_ratio(a, b):
    a = set(a)
    b = set(b)
    if not a or not b:
        return 0.0
    return len(a & b) / max(1, min(len(a), len(b)))

def extract_years(text):
    text = clean_text(text).lower()
    nums = []
    for pat in [r'(\d+)\s*лет', r'(\d+)\s*года', r'(\d+)\s*год', r'от\s*(\d+)', r'более\s*(\d+)']:
        nums.extend([int(x) for x in re.findall(pat, text)])
    return max(nums) if nums else 0

def startswith_match(a, b):
    a = clean_text(a).lower()
    b = clean_text(b).lower()
    if not a or not b:
        return 0.0
    return 1.0 if a.startswith(b) or b.startswith(a) else 0.0

def feature_row(resume_text, vacancy_text):
    r_text = clean_text(resume_text)
    v_text = clean_text(vacancy_text)

    r_title = extract_field(r_text, FIELD_ALIASES['resume_title'])
    v_title = extract_field(v_text, FIELD_ALIASES['vacancy_title'])
    r_skills = extract_field(r_text, FIELD_ALIASES['resume_skills'])
    v_skills = extract_field(v_text, FIELD_ALIASES['vacancy_skills'])
    r_exp = extract_field(r_text, FIELD_ALIASES['resume_exp'])
    v_exp = extract_field(v_text, FIELD_ALIASES['vacancy_exp'])
    r_region = extract_field(r_text, FIELD_ALIASES['resume_region'])
    v_region = extract_field(v_text, FIELD_ALIASES['vacancy_region'])
    r_sphere = extract_field(r_text, FIELD_ALIASES['resume_sphere'])
    v_sphere = extract_field(v_text, FIELD_ALIASES['vacancy_sphere'])

    r_tokens = tokenize(r_text)
    v_tokens = tokenize(v_text)
    r_title_toks = tokenize(r_title)
    v_title_toks = tokenize(v_title)
    r_skill_toks = tokenize(r_skills)
    v_skill_toks = tokenize(v_skills)
    r_sphere_toks = tokenize(r_sphere)
    v_sphere_toks = tokenize(v_sphere)

    r_years = extract_years(r_exp)
    v_years = extract_years(v_exp)
    years_gap = abs(r_years - v_years) if (r_years > 0 and v_years > 0) else 99

    return {
        'token_jaccard': jaccard(r_tokens, v_tokens),
        'token_overlap_ratio': overlap_ratio(r_tokens, v_tokens),
        'title_jaccard': jaccard(r_title_toks, v_title_toks),
        'title_overlap_ratio': overlap_ratio(r_title_toks, v_title_toks),
        'title_prefix_match': startswith_match(r_title, v_title),
        'skills_jaccard': jaccard(r_skill_toks, v_skill_toks),
        'skills_overlap_ratio': overlap_ratio(r_skill_toks, v_skill_toks),
        'sphere_jaccard': jaccard(r_sphere_toks, v_sphere_toks),
        'exact_region_match': 1.0 if clean_text(r_region).lower() and clean_text(r_region).lower() == clean_text(v_region).lower() else 0.0,
        'region_token_overlap': jaccard(tokenize(r_region), tokenize(v_region)),
        'r_years': float(r_years),
        'v_years': float(v_years),
        'years_gap': float(years_gap),
        'years_match_0': 1.0 if years_gap == 0 else 0.0,
        'years_match_1': 1.0 if years_gap <= 1 else 0.0,
        'resume_len_tokens': float(len(r_tokens)),
        'vacancy_len_tokens': float(len(v_tokens)),
    }


## 3. TF-IDF как дополнительный признак

In [ ]:
train_corpus = pd.concat([train_df['resume_text'].map(clean_text), train_df['vacancy_text'].map(clean_text)], axis=0).tolist()
tfidf_vectorizer = TfidfVectorizer(min_df=1, ngram_range=(1, 2), max_features=50000)
tfidf_vectorizer.fit(train_corpus)

def tfidf_score(resume_text, vacancy_text):
    rv = tfidf_vectorizer.transform([clean_text(resume_text)])
    vv = tfidf_vectorizer.transform([clean_text(vacancy_text)])
    return float(cosine_similarity(rv, vv)[0, 0])


## 4. Формирование таблицы признаков

In [ ]:
def add_model_features(df, bi_model, cross_model):
    rows = []
    for _, r in df.iterrows():
        resume_text = clean_text(r['resume_text'])
        vacancy_text = clean_text(r['vacancy_text'])

        q_emb = bi_model.encode([f'query: {resume_text}'], convert_to_numpy=True, normalize_embeddings=True)[0]
        d_emb = bi_model.encode([f'passage: {vacancy_text}'], convert_to_numpy=True, normalize_embeddings=True)[0]
        bi_score = float(np.dot(q_emb, d_emb))
        cross_score = float(cross_model.predict([[resume_text, vacancy_text]])[0])
        tfidf_sim = tfidf_score(resume_text, vacancy_text)

        feats = feature_row(resume_text, vacancy_text)
        rows.append({
            'resume_id': r['resume_id'],
            'vacancy_id': r['vacancy_id'],
            'label': int(r['label']),
            'bi_score': bi_score,
            'cross_score': cross_score,
            'tfidf_score': tfidf_sim,
            **feats,
        })
    return pd.DataFrame(rows)

train_feat = add_model_features(train_df, bi_model, cross_model)
val_feat = add_model_features(val_df, bi_model, cross_model)
test_feat = add_model_features(test_df, bi_model, cross_model)

train_feat.head()


,resume_id,vacancy_id,label,bi_score,cross_score,tfidf_score,token_jaccard,token_overlap_ratio,title_jaccard,title_overlap_ratio,...,sphere_jaccard,exact_region_match,region_token_overlap,r_years,v_years,years_gap,years_match_0,years_match_1,resume_len_tokens,vacancy_len_tokens
0,000a4860-4725-11ee-91de-01e20877b34e,f822e22a-0eb1-11ee-8ec2-0d972cff014f,1,0.675351,0.043409,0.049844,0.116667,0.218750,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,99.0,0.0,0.0,43.0,54.0
1,000a4860-4725-11ee-91de-01e20877b34e,744b3700-e3ca-11ef-86ee-d549be31d974,0,0.740725,0.045537,0.102511,0.080460,0.218750,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,99.0,0.0,0.0,43.0,107.0
2,000a4860-4725-11ee-91de-01e20877b34e,357ff596-afd4-11ee-9b60-cb26dff57dd7,0,0.900316,0.065661,0.100039,0.101266,0.250000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,99.0,0.0,0.0,43.0,105.0
3,000a4860-4725-11ee-91de-01e20877b34e,78780a02-4203-11eb-94f4-bf2cfe8c828d,0,0.660982,0.060471,0.088754,0.052632,0.107143,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,99.0,0.0,0.0,43.0,49.0
4,000a4860-4725-11ee-91de-01e20877b34e,7fe35d08-ad5f-11ef-a6ba-e7d0d2cf29b1,0,0.692457,0.027622,0.056178,0.048387,0.093750,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,99.0,0.0,0.0,43.0,59.0


## 5. Обучение мета-моделей

In [ ]:
FEATURES = [
    'bi_score', 'cross_score', 'tfidf_score',
    'token_jaccard', 'token_overlap_ratio',
    'title_jaccard', 'title_overlap_ratio', 'title_prefix_match',
    'skills_jaccard', 'skills_overlap_ratio',
    'sphere_jaccard', 'exact_region_match', 'region_token_overlap',
    'r_years', 'v_years', 'years_gap', 'years_match_0', 'years_match_1',
    'resume_len_tokens', 'vacancy_len_tokens'
]

X_train = train_feat[FEATURES].fillna(0.0).values
y_train = train_feat['label'].values
X_val = val_feat[FEATURES].fillna(0.0).values
y_val = val_feat['label'].values
X_test = test_feat[FEATURES].fillna(0.0).values
y_test = test_feat['label'].values

meta_logreg = LogisticRegression(class_weight='balanced', max_iter=3000, C=0.5, random_state=42)
meta_logreg.fit(X_train, y_train)

meta_hgb = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_depth=4,
    max_iter=300,
    min_samples_leaf=10,
    random_state=42,
)
meta_hgb.fit(X_train, y_train)

coef_df = pd.DataFrame({'feature': FEATURES, 'coef': meta_logreg.coef_[0]}).sort_values('coef', ascending=False)
coef_df


,feature,coef
0,bi_score,5.647560
1,cross_score,3.383020
5,title_jaccard,0.421929
14,v_years,0.226924
7,title_prefix_match,0.209153
6,title_overlap_ratio,0.119817
18,resume_len_tokens,0.007310
19,vacancy_len_tokens,0.003827
17,years_match_1,0.000000
13,r_years,0.000000


## 6. Ранжирование и метрики

In [ ]:
def rank_from_scores(eval_feat_df, score_col):
    rows = []
    for resume_id, group in eval_feat_df.groupby('resume_id'):
        group = group.sort_values(score_col, ascending=False)
        ranked_vacancy_ids = group['vacancy_id'].tolist()
        ranked_relevances = group['label'].astype(int).tolist()
        total_relevant = int(group['label'].sum())
        rows.append({
            'resume_id': resume_id,
            'ranked_vacancy_ids': ranked_vacancy_ids,
            'relevances': ranked_relevances,
            'Recall@3': recall_at_k(ranked_relevances, total_relevant, 3),
            'Recall@5': recall_at_k(ranked_relevances, total_relevant, 5),
            'RR': reciprocal_rank(ranked_relevances),
            'NDCG@5': ndcg_at_k(ranked_relevances, 5),
        })
    return pd.DataFrame(rows)

def add_meta_scores(feat_df, logreg_model, hgb_model):
    feat_df = feat_df.copy()
    X = feat_df[FEATURES].fillna(0.0).values
    feat_df['hybrid_logreg_score'] = logreg_model.predict_proba(X)[:, 1]
    feat_df['hybrid_hgb_score'] = hgb_model.predict_proba(X)[:, 1]
    return feat_df

val_feat_scored = add_meta_scores(val_feat, meta_logreg, meta_hgb)
test_feat_scored = add_meta_scores(test_feat, meta_logreg, meta_hgb)

test_bi = rank_from_scores(test_feat_scored, 'bi_score')
test_cross = rank_from_scores(test_feat_scored, 'cross_score')
test_logreg = rank_from_scores(test_feat_scored, 'hybrid_logreg_score')
test_hgb = rank_from_scores(test_feat_scored, 'hybrid_hgb_score')

test_comparison_df = pd.DataFrame([
    summarize_metrics(test_bi, 'bi-encoder'),
    summarize_metrics(test_cross, 'cross-encoder'),
    summarize_metrics(test_logreg, 'hybrid logreg'),
    summarize_metrics(test_hgb, 'hybrid hgb'),
]).sort_values(['MRR','NDCG@5'], ascending=False).reset_index(drop=True)
test_comparison_df


,model,Recall@3,Recall@5,MRR,NDCG@5
0,hybrid hgb,0.789474,0.894737,0.680808,0.726180
1,hybrid logreg,0.684211,0.842105,0.629156,0.665316
2,cross-encoder,0.631579,0.631579,0.551777,0.540098
3,bi-encoder,0.684211,0.736842,0.547391,0.569656


## 7. Проверка на validation

In [ ]:
val_bi = rank_from_scores(val_feat_scored, 'bi_score')
val_cross = rank_from_scores(val_feat_scored, 'cross_score')
val_logreg = rank_from_scores(val_feat_scored, 'hybrid_logreg_score')
val_hgb = rank_from_scores(val_feat_scored, 'hybrid_hgb_score')

val_comparison_df = pd.DataFrame([
    summarize_metrics(val_bi, 'bi-encoder'),
    summarize_metrics(val_cross, 'cross-encoder'),
    summarize_metrics(val_logreg, 'hybrid logreg'),
    summarize_metrics(val_hgb, 'hybrid hgb'),
]).sort_values(['MRR','NDCG@5'], ascending=False).reset_index(drop=True)
val_comparison_df


,model,Recall@3,Recall@5,MRR,NDCG@5
0,bi-encoder,0.789474,0.842105,0.736800,0.753867
1,hybrid logreg,0.789474,0.789474,0.726480,0.724308
2,hybrid hgb,0.842105,0.842105,0.709169,0.731199
3,cross-encoder,0.526316,0.736842,0.486656,0.525475


## 8. Проверка hybrid

In [ ]:
per_resume_compare = (
    test_bi[['resume_id','ranked_vacancy_ids','relevances','Recall@3','Recall@5','RR','NDCG@5']]
    .rename(columns={'ranked_vacancy_ids':'bi_ranked_vacancy_ids','relevances':'bi_relevances','Recall@3':'bi_Recall@3','Recall@5':'bi_Recall@5','RR':'bi_RR','NDCG@5':'bi_NDCG@5'})
    .merge(
        test_cross[['resume_id','ranked_vacancy_ids','relevances','Recall@3','Recall@5','RR','NDCG@5']].rename(columns={'ranked_vacancy_ids':'cross_ranked_vacancy_ids','relevances':'cross_relevances','Recall@3':'cross_Recall@3','Recall@5':'cross_Recall@5','RR':'cross_RR','NDCG@5':'cross_NDCG@5'}),
        on='resume_id'
    )
    .merge(
        test_hgb[['resume_id','ranked_vacancy_ids','relevances','Recall@3','Recall@5','RR','NDCG@5']].rename(columns={'ranked_vacancy_ids':'hybrid_ranked_vacancy_ids','relevances':'hybrid_relevances','Recall@3':'hybrid_Recall@3','Recall@5':'hybrid_Recall@5','RR':'hybrid_RR','NDCG@5':'hybrid_NDCG@5'}),
        on='resume_id'
    )
)
per_resume_compare['delta_vs_bi_RR'] = per_resume_compare['hybrid_RR'] - per_resume_compare['bi_RR']
per_resume_compare['delta_vs_cross_RR'] = per_resume_compare['hybrid_RR'] - per_resume_compare['cross_RR']
per_resume_compare.sort_values(['delta_vs_bi_RR','delta_vs_cross_RR'], ascending=False)


,resume_id,bi_ranked_vacancy_ids,bi_relevances,bi_Recall@3,bi_Recall@5,bi_RR,bi_NDCG@5,cross_ranked_vacancy_ids,cross_relevances,cross_Recall@3,...,cross_RR,cross_NDCG@5,hybrid_ranked_vacancy_ids,hybrid_relevances,hybrid_Recall@3,hybrid_Recall@5,hybrid_RR,hybrid_NDCG@5,delta_vs_bi_RR,delta_vs_cross_RR
15,bf0f2af0-87d9-11ef-936f-716c765c9fe3,"[45543eb0-e94c-11ef-a95d-e7d0d2cf29b1, b27f0f4...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]",0.0,0.0,0.083333,0.000000,"[ab5d0bf6-81bc-11ed-bc26-05dc90903fb8, 48691e8...","[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0]",0.0,...,0.125000,0.00000,"[e74a1215-0751-11ed-b057-57fc951f3846, 45543eb...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1.0,1.0,1.000000,1.000000,0.916667,0.875000
10,6fded420-3e30-11ed-9fb0-fdf9f86d256a,"[e84f0455-6160-11ec-9c40-57fc951f3846, ea41f8c...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]",0.0,0.0,0.090909,0.000000,"[38da7968-e2fd-11ef-a141-cb26dff57dd7, a039ecc...","[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1.0,...,0.333333,0.50000,"[f11de0e5-d0e8-11ec-8c63-57fc951f3846, e84f045...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1.0,1.0,1.000000,1.000000,0.909091,0.666667
7,43dac4e0-bb2b-11ea-8246-e736a3d3ed84,"[ab5d0bf6-81bc-11ed-bc26-05dc90903fb8, f890ad3...","[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]",0.0,0.0,0.166667,0.000000,"[f890ad35-509e-11ee-a2eb-9586bb63c653, 934e492...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]",0.0,...,0.090909,0.00000,"[3f957b95-c6c9-11ec-a9c3-5b1b13d7f12f, ab5d0bf...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1.0,1.0,1.000000,1.000000,0.833333,0.909091
17,d9f2adb0-38cb-11ed-91b0-7fb917d16256,"[6e2ebd78-24a7-11ef-bb5e-e7d0d2cf29b1, d558e54...","[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1.0,1.0,0.333333,0.500000,"[6e2ebd78-24a7-11ef-bb5e-e7d0d2cf29b1, 830067f...","[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0]",0.0,...,0.125000,0.00000,"[4c809cc8-9d01-11ef-aaa4-e7d0d2cf29b1, 6e2ebd7...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1.0,1.0,1.000000,1.000000,0.666667,0.875000
13,9765c9b0-dba2-11ef-aad2-e7f46f7514cd,"[50ab53f0-f975-11ef-b8fe-e7d0d2cf29b1, 122776e...","[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1.0,1.0,0.333333,0.500000,"[29aec315-8377-11ee-8ffb-d549be31d974, e593060...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1.0,...,1.000000,1.00000,"[29aec315-8377-11ee-8ffb-d549be31d974, 122776e...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1.0,1.0,1.000000,1.000000,0.666667,0.000000
14,a3b948a0-51a7-11ec-b050-1b29d3b53cbb,"[b3db8775-89c7-11ee-94a2-d549be31d974, c48694a...","[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1.0,1.0,0.500000,0.630930,"[f2eaa268-8788-11ef-9fe0-d549be31d974, 01ec1c6...","[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1.0,...,0.333333,0.50000,"[c48694a8-d950-11ef-9776-cb26dff57dd7, af27882...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1.0,1.0,1.000000,1.000000,0.500000,0.666667
4,283eb8b0-ab2a-11ef-979e-c7bd3a8c3ec7,"[a5f0b120-15f9-11f0-add8-cb26dff57dd7, 24e894f...","[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]",0.0,0.0,0.142857,0.000000,"[a5f0b120-15f9-11f0-add8-cb26dff57dd7, e651c70...","[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0]",0.0,...,0.125000,0.00000,"[24e894f5-00a9-11ef-8b1d-e7d0d2cf29b1, b4c1226...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",0.0,1.0,0.200000,0.386853,0.057143,0.075000
3,20f2d640-310f-11ee-a699-d53cd48f29e6,"[78780a02-4203-11eb-94f4-bf2cfe8c828d, 17c92e9...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1.0,1.0,1.000000,1.000000,"[ebcc4235-74fe-11ec-833e-4febb26dc4ec, 37c5b0e...","[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]",0.0,...,0.142857,0.00000,"[78780a02-4203-11eb-94f4-bf2cfe8c828d, ebcc423...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1.0,1.0,1.000000,1.000000,0.000000,0.857143
5,2ee276a5-a364-11e7-ad25-037acc02728d,"[0a8467d8-b862-11ef-bd15-cb26dff57dd7, 7a58327...","[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1.0,1.0,0.333333,0.500000,"[b6909c60-1438-11f0-a313-d549be31d974, 38da796...","[0

## 9. Repeated evaluation по нескольким seed


In [ ]:
REPEATED_SEEDS = [42, 52, 62, 72, 82]
BOOTSTRAP_FRAC = 1.0
USE_BOOTSTRAP = True


In [ ]:
def bootstrap_train_df(train_feat_df, seed=42, frac=1.0):
    rng = np.random.default_rng(seed)
    sampled_parts = []
    for resume_id, group in train_feat_df.groupby('resume_id'):
        n = max(1, int(round(len(group) * frac)))
        idx = rng.choice(group.index.to_numpy(), size=n, replace=True)
        sampled_parts.append(train_feat_df.loc[idx])
    out = pd.concat(sampled_parts, axis=0).reset_index(drop=True)
    return out

def fit_meta_models(train_feat_df, seed=42):
    X_train = train_feat_df[FEATURES].fillna(0.0).values
    y_train = train_feat_df['label'].values

    logreg = LogisticRegression(
        class_weight='balanced',
        max_iter=3000,
        C=0.5,
        random_state=seed,
    )
    logreg.fit(X_train, y_train)

    hgb = HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_depth=4,
        max_iter=300,
        min_samples_leaf=10,
        random_state=seed,
    )
    hgb.fit(X_train, y_train)
    return logreg, hgb

def score_meta_models(feat_df, logreg_model, hgb_model):
    feat_df = feat_df.copy()
    X = feat_df[FEATURES].fillna(0.0).values
    feat_df['hybrid_logreg_score_rep'] = logreg_model.predict_proba(X)[:, 1]
    feat_df['hybrid_hgb_score_rep'] = hgb_model.predict_proba(X)[:, 1]
    return feat_df


In [ ]:
repeated_rows = []

for seed in REPEATED_SEEDS:
    train_feat_rep = bootstrap_train_df(train_feat, seed=seed, frac=BOOTSTRAP_FRAC) if USE_BOOTSTRAP else train_feat.copy()
    logreg_rep, hgb_rep = fit_meta_models(train_feat_rep, seed=seed)

    val_feat_rep = score_meta_models(val_feat, logreg_rep, hgb_rep)
    test_feat_rep = score_meta_models(test_feat, logreg_rep, hgb_rep)

    val_logreg_rep = rank_from_scores(val_feat_rep.rename(columns={'hybrid_logreg_score_rep': 'hybrid_logreg_score'}), 'hybrid_logreg_score')
    val_hgb_rep = rank_from_scores(val_feat_rep.rename(columns={'hybrid_hgb_score_rep': 'hybrid_hgb_score'}), 'hybrid_hgb_score')
    test_logreg_rep = rank_from_scores(test_feat_rep.rename(columns={'hybrid_logreg_score_rep': 'hybrid_logreg_score'}), 'hybrid_logreg_score')
    test_hgb_rep = rank_from_scores(test_feat_rep.rename(columns={'hybrid_hgb_score_rep': 'hybrid_hgb_score'}), 'hybrid_hgb_score')

    for split_name, details_df, model_name in [
        ('val', val_logreg_rep, 'hybrid logreg'),
        ('val', val_hgb_rep, 'hybrid hgb'),
        ('test', test_logreg_rep, 'hybrid logreg'),
        ('test', test_hgb_rep, 'hybrid hgb'),
    ]:
        metrics = summarize_metrics(details_df, model_name)
        repeated_rows.append({'seed': seed, 'split': split_name, **metrics})

repeated_df = pd.DataFrame(repeated_rows)
repeated_df.head()


,seed,split,model,Recall@3,Recall@5,MRR,NDCG@5
0,42,val,hybrid logreg,0.789474,0.789474,0.727733,0.724308
1,42,val,hybrid hgb,0.736842,0.894737,0.614181,0.674485
2,42,test,hybrid logreg,0.684211,0.842105,0.649332,0.682434
3,42,test,hybrid hgb,0.736842,0.842105,0.698371,0.719289
4,52,val,hybrid logreg,0.789474,0.789474,0.663847,0.678568


## 10. Сводка по repeated evaluation


In [ ]:
summary_tables = []
for split_name in ['val', 'test']:
    for model_name in ['hybrid logreg', 'hybrid hgb']:
        part = repeated_df[(repeated_df['split'] == split_name) & (repeated_df['model'] == model_name)]
        if len(part) == 0:
            continue
        summary_tables.append({
            'split': split_name,
            'model': model_name,
            'Recall@3_mean': part['Recall@3'].mean(),
            'Recall@3_std': part['Recall@3'].std(ddof=1),
            'Recall@5_mean': part['Recall@5'].mean(),
            'Recall@5_std': part['Recall@5'].std(ddof=1),
            'MRR_mean': part['MRR'].mean(),
            'MRR_std': part['MRR'].std(ddof=1),
            'NDCG@5_mean': part['NDCG@5'].mean(),
            'NDCG@5_std': part['NDCG@5'].std(ddof=1),
            'MRR_min': part['MRR'].min(),
            'MRR_max': part['MRR'].max(),
            'NDCG@5_min': part['NDCG@5'].min(),
            'NDCG@5_max': part['NDCG@5'].max(),
        })

repeated_summary_df = pd.DataFrame(summary_tables)
repeated_summary_df


,split,model,Recall@3_mean,Recall@3_std,Recall@5_mean,Recall@5_std,MRR_mean,MRR_std,NDCG@5_mean,NDCG@5_std,MRR_min,MRR_max,NDCG@5_min,NDCG@5_max
0,val,hybrid logreg,0.778947,0.023538,0.789474,0.000000,0.709818,0.026026,0.711674,0.018724,0.663847,0.727733,0.678568,0.724308
1,val,hybrid hgb,0.726316,0.068623,0.831579,0.057655,0.647113,0.078030,0.678107,0.069281,0.559667,0.767943,0.580819,0.773291
2,test,hybrid logreg,0.715789,0.047075,0.863158,0.047075,0.672598,0.043861,0.706517,0.040619,0.626901,0.735088,0.665316,0.755426
3,test,hybrid hgb,0.715789,0.060009,0.863158,0.028828,0.656921,0.037662,0.694650,0.035096,0.617251,0.698371,0.656430,0.728487
